In [1]:
from nanover.recording import NanoverRecordingReader
from tutorials.experiments.keyframes import extract_keyframes

IN_PATH = "knot-tying-checkpoints-2.nanover.zip"
# IN_PATH = "trivial-checkpoints.nanover.zip"
IN_PATH = "knot-tying-checkpoints-better.nanover.zip"

with NanoverRecordingReader.from_path(IN_PATH) as reader:
    KEYFRAMES = extract_keyframes(reader)

KEYFRAMES

[KeyFrame(targets=[TargetGroup(particles=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11], position=array([3.59112048, 1.87643147, 5.51278114]), centroid=array([3.6178942, 1.8746958, 5.5276   ], dtype=float32)), TargetGroup(particles=[162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172], position=array([4.52477264, 6.04662466, 1.39771509]), centroid=array([4.4619384, 5.978038 , 1.4505223], dtype=float32))], centroids=array([[3.6178942, 1.8746958, 5.5276   ],
        [4.4619384, 5.978038 , 1.4505223]], dtype=float32)),
 KeyFrame(targets=[TargetGroup(particles=[42, 43, 44, 45, 46, 47, 48, 49, 50, 51], position=array([2.77324629, 4.49306536, 2.95724869]), centroid=array([2.7566574, 4.4498696, 2.9436188], dtype=float32)), TargetGroup(particles=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11], position=array([2.80388904, 4.27558708, 1.47705877]), centroid=array([2.798412 , 4.247428 , 1.4613428], dtype=float32)), TargetGroup(particles=[72, 73, 74, 75, 76, 77, 78, 79, 80, 81], position=array([3.57914352, 3.433

In [2]:
## Setup runner
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="checkpoint replay")
imd_runner.load(0)

In [3]:
simulation.use_pbc_wrapping = False

In [4]:
from nanover.websocket import NanoverImdClient

client = NanoverImdClient.from_runner(imd_runner)
with client.root_selection.modify() as selection:
    selection.renderer = "cartoon"

In [5]:
def notify_all(message):
    for command in imd_runner.app_server.commands:
        if "/notify" in command:
            imd_runner.app_server.run_command(command, dict(message=message))

In [6]:
from smearmd import SmearAgent

KEYFRAME_INDEX = 0
AGENT: SmearAgent | None = None


def smear_next():
    global KEYFRAME_INDEX
    KEYFRAME_INDEX = min(len(KEYFRAMES) - 1, KEYFRAME_INDEX + 1)

    notify_all(f"MATCHING KEYFRAME {KEYFRAME_INDEX}")
    smear_match(KEYFRAMES[KEYFRAME_INDEX])


def smear_back():
    global KEYFRAME_INDEX
    KEYFRAME_INDEX = max(0, KEYFRAME_INDEX - 1)

    notify_all(f"MATCHING KEYFRAME {KEYFRAME_INDEX}")
    smear_match(KEYFRAMES[KEYFRAME_INDEX])


def smear_match(keyframe):
    global AGENT

    if AGENT is not None:
        AGENT.stop()

    AGENT = SmearAgent.from_runner(imd_runner)
    AGENT.speed = 5
    AGENT.keyframe = keyframe
    AGENT.start()


def smear_cancel():
    global AGENT, KEYFRAME_INDEX
    AGENT.stop()
    AGENT = None
    KEYFRAME_INDEX = 0

    notify_all(f"RESETTING AGENT")


imd_runner.app_server.register_command("user/smear/next", smear_next)
imd_runner.app_server.register_command("user/smear/back", smear_back)
imd_runner.app_server.register_command("user/smear/cancel", smear_cancel)

In [7]:
from ipywidgets import Output
output = Output()
output

Output()

In [8]:
with output:
    print("test")

In [9]:
from nanover.jupyter.nglclient import NGLClient

client = NGLClient.from_runner(imd_runner)
client.view

NGLWidget()

In [11]:
from nanover.jupyter.controls import show_runner_controls

show_runner_controls(imd_runner)

Button(description='Close Server', style=ButtonStyle())

interactive(children=(Dropdown(description='Force Type', index=1, options=(('Gaussian', 'gaussian'), ('Harmoni…

interactive(children=(IntSlider(value=100, description='Force Scale', max=1000, min=1), Output()), _dom_classe…

interactive(children=(FloatSlider(value=0.4, description='Force Range', max=2.0, min=0.1, step=0.05), Output()…

interactive(children=(FloatSlider(value=1.0, description='Passthrough', max=1.0), Output()), _dom_classes=('wi…